In [7]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"

excel_files = sorted(
    path
    for path in RAW_DIR.glob("*.xlsx")
    if not path.name.startswith("~$")
)

if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {RAW_DIR}")

for index, path in enumerate(excel_files):
    print(f"{index}: {path.name}")

# Change this index if the intended workbook isn't first.
DATA_PATH = excel_files[0]
print("\nSelected:", DATA_PATH.name)

0: July 2026 Complete Monthly Ridership (with adjustments and estimates)_260901.xlsx

Selected: July 2026 Complete Monthly Ridership (with adjustments and estimates)_260901.xlsx


In [8]:
NTD_ID = "00001"  # King County
MODE = "MB"       # Motorbus
TOS = "DO"        # Directly operated

upt = pd.read_excel(
    DATA_PATH,
    sheet_name="UPT",
    dtype={"NTD ID": "string"},
)

seattle = upt.loc[
    upt["NTD ID"].eq(NTD_ID)
    & upt["Mode"].eq(MODE)
    & upt["TOS"].eq(TOS)
].copy()

if len(seattle) != 1:
    raise ValueError(
        f"Expected one matching Seattle row, found {len(seattle)}."
    )

display(seattle[["NTD ID", "Agency", "Mode", "TOS"]])

,NTD ID,Agency,Mode,TOS
4,00001,King County,MB,DO


In [9]:
# Identify month columns such as "1/2002" and "7/2026".
month_columns = [
    column
    for column in upt.columns
    if pd.Series([str(column)]).str.fullmatch(
        r"\d{1,2}/\d{4}"
    ).iloc[0]
]

monthly = seattle.melt(
    value_vars=month_columns,
    var_name="date",
    value_name="ridership",
)

monthly["date"] = pd.to_datetime(
    monthly["date"],
    format="%m/%Y",
)

monthly["ridership"] = pd.to_numeric(
    monthly["ridership"],
    errors="raise",
)

monthly = (
    monthly
    .set_index("date")
    .sort_index()
    .asfreq("MS")  # One observation per calendar month.
)

print(f"Number of months: {len(monthly)}")
print(f"Missing ridership values: {monthly['ridership'].isna().sum()}")

display(monthly.head())
display(monthly.tail())

Number of months: 295
Missing ridership values: 0


,ridership
date,
2002-01-01,6045861.0
2002-02-01,5406135.0
2002-03-01,5999230.0
2002-04-01,6058398.0
2002-05-01,6134503.0


,ridership
date,
2026-03-01,6646228.0
2026-04-01,6738016.0
2026-05-01,6932761.0
2026-06-01,6611961.0
2026-07-01,6298063.0


In [10]:
# Check that the cleaned dataset is ready to save.
if monthly.empty:
    raise ValueError("The monthly dataset is empty.")

if monthly.index.has_duplicates:
    raise ValueError("Duplicate months found.")

if not monthly.index.is_monotonic_increasing:
    raise ValueError("Months are not in chronological order.")

expected_months = pd.date_range(
    start=monthly.index.min(),
    end=monthly.index.max(),
    freq="MS",
)

if not monthly.index.equals(expected_months):
    raise ValueError("The date index has missing or unexpected months.")

if monthly["ridership"].isna().any():
    raise ValueError("Missing ridership values found.")

if monthly["ridership"].lt(0).any():
    raise ValueError("Negative ridership values found.")

print(f"Data checks passed: {len(monthly)} months ready to export.")

Data checks passed: 295 months ready to export.


In [11]:
# Keep the original monthly dataset unchanged.
features = monthly[["ridership"]].copy()

features["ridership_lag_1"] = features["ridership"].shift(1)
features["ridership_lag_12"] = features["ridership"].shift(12)

# Shift first so the average excludes the month we're predicting.
features["rolling_mean_3"] = (
    features["ridership"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

display(features.tail(12))
print("Missing values from the required history:")
display(features.isna().sum())

,ridership,ridership_lag_1,ridership_lag_12,rolling_mean_3
date,,,,
2025-08-01,6629625.0,6716230.0,6019974.0,6.792834e+06
2025-09-01,6597525.0,6629625.0,6139353.0,6.678241e+06
2025-10-01,7027582.0,6597525.0,6820828.0,6.647793e+06
2025-11-01,6034095.0,7027582.0,5983373.0,6.751577e+06
2025-12-01,5848180.0,6034095.0,5655460.0,6.553067e+06
2026-01-01,6315539.0,5848180.0,6158381.0,6.303286e+06
2026-02-01,5912095.0,6315539.0,5748810.0,6.065938e+06
2026-03-01,6646228.0,5912095.0,6601733.0,6.025271e+06
2026-04-01,6738016.0,6646228.0,6841791.0,6.291287e+06


Missing values from the required history:


ridership            0
ridership_lag_1      1
ridership_lag_12    12
rolling_mean_3       3
dtype: int64

In [ ]:
# Calendar information is known ahead of time
features["month"] = features.index.month
features["quarter"] = features.index.quarter
features["days_in_month"] = features.index.days_in_month

display(
    features[
        ["ridership", "month", "quarter", "days_in_month"]
    ].tail(12)
)

In [12]:
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / (
    f"ntd_{NTD_ID}_{MODE}_{TOS}_monthly.csv"
)

monthly.to_csv(output_path, index_label="date")

print(f"Saved {len(monthly)} months to {output_path}")

Saved 295 months to /Users/alliesewell/transit_dfc_mlpl/data/processed/ntd_00001_MB_DO_monthly.csv
